In [ ]:
!pip install -q \
langgraph \
langchain \
langchain-community \
langchain-ollama \
streamlit \
pandas

In [1]:
from langchain_ollama import ChatOllama

# ── Speed optimizations ──────────────────────────────────────────────────────
# num_predict : hard cap on output tokens — stops runaway generation on CPU
# num_ctx     : smaller context window → less RAM, faster prefill
# num_thread  : use all physical cores for token generation
# repeat_penalty: avoids repetitive loops that waste tokens
# ─────────────────────────────────────────────────────────────────────────────
llm = ChatOllama(
    model="gemma3:12b",
    temperature=0.2,
    num_predict=300,   # cap output — agents need ~150-250 tokens, not 2000
    num_ctx=2048,      # shrink context window; saves huge RAM/compute on CPU
    num_thread=8,      # set to your physical core count (adjust if needed)
    repeat_penalty=1.1,
)

print(llm.invoke("Reply with READY").content)


READY



In [2]:
intel_report = """
Timestamp: 2026-06-15 14:30 UTC

ISR Drone Feed:

Detected movement of 12 armored vehicles
near Northern Sector Delta.

Satellite imagery indicates
temporary encampment established
within the last 24 hours.

Communications intercepts show
encrypted radio traffic increased
by 240%.

Weather conditions favorable
for vehicle movement.

Nearby civilian settlements:
3 villages within 12km radius.
"""

In [3]:
def run_agent(role, task):
    """Invoke LLM for a given role. Prompt kept short to reduce token budget."""
    prompt = (
        f"You are {role}.\n\n"
        f"Context:\n{task}\n\n"
        "Reply in military briefing style. Cover:\n"
        "1. Findings  2. Confidence (0-100)  3. Concerns  4. Recommendation\n"
        "Be concise."
    )
    return llm.invoke(prompt).content


In [4]:
from typing import TypedDict

class MissionState(TypedDict):

    report: str

    isr_output: str

    cyber_output: str

    logistics_output: str

    threat_output: str

    planner_output: str

    governance_output: str

    commander_brief: str

In [5]:
def isr_agent(state):

    result = run_agent(
        "ISR Intelligence Officer",
        state["report"]
    )

    return {
        "isr_output": result
    }

In [6]:
def cyber_agent(state):

    result = run_agent(
        "Cyber Intelligence Officer",
        state["report"]
    )

    return {
        "cyber_output": result
    }

In [9]:
def logistics_agent(state):

    logistics_prompt = f"""
Mission Report:

{state['report']}

Assume current operational readiness.

Assess:

- fuel
- supply chain
- mobility
- medical readiness

Identify possible logistics risks.
"""

    result = run_agent(
        "Logistics Officer",
        logistics_prompt
    )

    return {
        "logistics_output": result
    }

In [10]:
def threat_agent(state):

    prompt = f"""
You are Threat Assessment Officer.

ISR Findings:
{state['isr_output']}

Cyber Findings:
{state['cyber_output']}

Logistics Findings:
{state['logistics_output']}

Determine:

GREEN
YELLOW
ORANGE
RED

Explain rationale.

Provide confidence score.
"""

    result = llm.invoke(prompt).content

    return {
        "threat_output": result
    }

In [11]:
def planner_agent(state):

    prompt = f"""
You are Mission Planning Officer.

Threat Assessment:

{state['threat_output']}

Generate 3 Courses Of Action.

For each:

- Objective
- Benefits
- Risks
- Resources Needed

Decision support only.
No autonomous actions.
"""

    result = llm.invoke(prompt).content

    return {
        "planner_output": result
    }

In [12]:
def governance_agent(state):

    prompt = f"""
You are Governance and Compliance Officer.

Review:

{state['planner_output']}

Check:

- policy compliance
- civilian impact
- escalation risk
- ethical concerns

Output:

APPROVED
or

REQUIRES HUMAN REVIEW

Explain reasoning.
"""

    result = llm.invoke(prompt).content

    return {
        "governance_output": result
    }

In [13]:
def debate_agent(state):

    prompt = f"""
Conduct an internal review.

ISR Officer:
{state['isr_output']}

Cyber Officer:
{state['cyber_output']}

Threat Officer:
{state['threat_output']}

Mission Planner:
{state['planner_output']}

Perform a debate:

1. Agreements
2. Disagreements
3. Missing evidence
4. Revised conclusions
"""

    result = llm.invoke(prompt).content

    return {
        "planner_output":
            state["planner_output"]
            + "\n\nDEBATE REVIEW\n\n"
            + result
    }

In [14]:
def commander_agent(state):

    prompt = f"""
Generate Executive Commander Brief.

Mission Report:
{state['report']}

ISR:
{state['isr_output']}

Cyber:
{state['cyber_output']}

Logistics:
{state['logistics_output']}

Threat:
{state['threat_output']}

Mission Plan:
{state['planner_output']}

Governance:
{state['governance_output']}

Format:

Situation Summary

Threat Level

Recommended COAs

Risks

Confidence

Required Human Decisions
"""

    result = llm.invoke(prompt).content

    return {
        "commander_brief": result
    }

## ⚡ Parallel Fan-Out: ISR + Cyber + Logistics run simultaneously

In [15]:
import concurrent.futures

def parallel_first_layer(state):
    """Run ISR, Cyber, Logistics in parallel — they are fully independent."""
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
        fut_isr  = ex.submit(isr_agent, state)
        fut_cyber = ex.submit(cyber_agent, state)
        fut_log  = ex.submit(logistics_agent, state)
        isr_res  = fut_isr.result()
        cyber_res = fut_cyber.result()
        log_res  = fut_log.result()
    return {**isr_res, **cyber_res, **log_res}


In [16]:
from langgraph.graph import StateGraph, END

builder = StateGraph(MissionState)

# Parallel first layer wrapped as single node
builder.add_node("parallel_intel", parallel_first_layer)
builder.add_node("threat",      threat_agent)
builder.add_node("planner",     planner_agent)
builder.add_node("debate",      debate_agent)
builder.add_node("governance",  governance_agent)
builder.add_node("commander",   commander_agent)

builder.set_entry_point("parallel_intel")
builder.add_edge("parallel_intel", "threat")
builder.add_edge("threat",      "planner")
builder.add_edge("planner",     "debate")
builder.add_edge("debate",      "governance")
builder.add_edge("governance",  "commander")
builder.add_edge("commander",   END)

graph = builder.compile()


In [ ]:
import time
t0 = time.time()
result = graph.invoke({"report": intel_report})
print(f"\n✅ Total pipeline time: {time.time()-t0:.1f}s")


In [ ]:
print("\n=== ISR ===\n")
print(result["isr_output"])

print("\n=== CYBER ===\n")
print(result["cyber_output"])

print("\n=== LOGISTICS ===\n")
print(result["logistics_output"])

print("\n=== THREAT ===\n")
print(result["threat_output"])

print("\n=== PLANNER ===\n")
print(result["planner_output"])

print("\n=== GOVERNANCE ===\n")
print(result["governance_output"])

print("\n=== COMMANDER BRIEF ===\n")
print(result["commander_brief"])

In [1]:
%%writefile app.py

import streamlit as st
import concurrent.futures
from typing import TypedDict
from langchain_ollama import ChatOllama
from langgraph.graph import StateGraph, END

# ── LLM ──────────────────────────────────────────────────────────────────────
@st.cache_resource
def get_llm():
    return ChatOllama(
        model="qwen2.5:0.5b",
        temperature=0.2,
        num_predict=300,
        num_ctx=2048,
        num_thread=8,
        repeat_penalty=1.1,
    )

# ── State ─────────────────────────────────────────────────────────────────────
class MissionState(TypedDict):
    report: str
    isr_output: str
    cyber_output: str
    logistics_output: str
    threat_output: str
    planner_output: str
    governance_output: str
    commander_brief: str

# ── Agents ────────────────────────────────────────────────────────────────────
def run_agent(role, task):
    llm = get_llm()
    prompt = (
        f"You are {role}.\n\nContext:\n{task}\n\n"
        "Reply in military briefing style. Cover:\n"
        "1. Findings  2. Confidence (0-100)  3. Concerns  4. Recommendation\n"
        "Be concise."
    )
    return llm.invoke(prompt).content

def isr_agent(state):
    return {"isr_output": run_agent("ISR Intelligence Officer", state["report"])}

def cyber_agent(state):
    return {"cyber_output": run_agent("Cyber Intelligence Officer", state["report"])}

def logistics_agent(state):
    prompt = (
        f"Mission Report:\n{state['report']}\n\n"
        "Assume current operational readiness.\n"
        "Assess: fuel, supply chain, mobility, medical readiness.\n"
        "Identify logistics risks."
    )
    return {"logistics_output": run_agent("Logistics Officer", prompt)}

def parallel_first_layer(state):
    llm = get_llm()
    with concurrent.futures.ThreadPoolExecutor(max_workers=3) as ex:
        f1 = ex.submit(isr_agent, state)
        f2 = ex.submit(cyber_agent, state)
        f3 = ex.submit(logistics_agent, state)
        return {**f1.result(), **f2.result(), **f3.result()}

def threat_agent(state):
    llm = get_llm()
    prompt = (
        f"You are Threat Assessment Officer.\n\n"
        f"ISR:\n{state['isr_output']}\n\n"
        f"Cyber:\n{state['cyber_output']}\n\n"
        f"Logistics:\n{state['logistics_output']}\n\n"
        "Rate threat: GREEN / YELLOW / ORANGE / RED. Explain. Give confidence score."
    )
    return {"threat_output": llm.invoke(prompt).content}

def planner_agent(state):
    llm = get_llm()
    prompt = (
        f"You are Mission Planning Officer.\n\nThreat:\n{state['threat_output']}\n\n"
        "Generate 3 Courses of Action. For each: Objective, Benefits, Risks, Resources."
    )
    return {"planner_output": llm.invoke(prompt).content}

def debate_agent(state):
    llm = get_llm()
    prompt = (
        f"Internal review.\nISR:\n{state['isr_output']}\nCyber:\n{state['cyber_output']}\n"
        f"Threat:\n{state['threat_output']}\nPlan:\n{state['planner_output']}\n\n"
        "1. Agreements  2. Disagreements  3. Missing evidence  4. Revised conclusions"
    )
    result = llm.invoke(prompt).content
    return {"planner_output": state["planner_output"] + "\n\nDEBATE REVIEW\n\n" + result}

def governance_agent(state):
    llm = get_llm()
    prompt = (
        f"You are Governance & Compliance Officer.\n\nReview:\n{state['planner_output']}\n\n"
        "Check: policy compliance, civilian impact, escalation risk, ethical concerns.\n"
        "Output: APPROVED or REQUIRES HUMAN REVIEW. Explain."
    )
    return {"governance_output": llm.invoke(prompt).content}

def commander_agent(state):
    llm = get_llm()
    prompt = (
        f"Generate Executive Commander Brief.\n\n"
        f"Report:\n{state['report']}\nISR:\n{state['isr_output']}\n"
        f"Cyber:\n{state['cyber_output']}\nLogistics:\n{state['logistics_output']}\n"
        f"Threat:\n{state['threat_output']}\nPlan:\n{state['planner_output']}\n"
        f"Governance:\n{state['governance_output']}\n\n"
        "Format: Situation Summary | Threat Level | Recommended COAs | Risks | Confidence | Required Human Decisions"
    )
    return {"commander_brief": llm.invoke(prompt).content}

# ── Graph ─────────────────────────────────────────────────────────────────────
@st.cache_resource
def build_graph():
    b = StateGraph(MissionState)
    b.add_node("parallel_intel", parallel_first_layer)
    b.add_node("threat",     threat_agent)
    b.add_node("planner",    planner_agent)
    b.add_node("debate",     debate_agent)
    b.add_node("governance", governance_agent)
    b.add_node("commander",  commander_agent)
    b.set_entry_point("parallel_intel")
    b.add_edge("parallel_intel", "threat")
    b.add_edge("threat",     "planner")
    b.add_edge("planner",    "debate")
    b.add_edge("debate",     "governance")
    b.add_edge("governance", "commander")
    b.add_edge("commander",  END)
    return b.compile()

# ── UI ────────────────────────────────────────────────────────────────────────
st.set_page_config(page_title="Joint Operations Center", layout="wide")
st.title("🛡️ Agentic AI Joint Operations Center")

report = st.text_area(
    "Mission Intelligence",
    height=300,
    placeholder="Enter mission intelligence report...",
)

if st.button("🚀 Run Analysis", disabled=not report.strip()):
    graph = build_graph()
    STAGES = ["parallel_intel", "threat", "planner", "debate", "governance", "commander"]
    STAGE_LABELS = {
        "parallel_intel": "ISR + Cyber + Logistics (parallel)",
        "threat": "Threat Assessment",
        "planner": "Mission Planning",
        "debate": "Debate Review",
        "governance": "Governance",
        "commander": "Commander Brief",
    }

    progress = st.progress(0)
    status   = st.empty()
    result   = {}

    with st.spinner("Running multi-agent analysis pipeline..."):
        for i, step in enumerate(graph.stream({"report": report})):
            node_name = list(step.keys())[0]
            result.update(step[node_name])
            status.text(f"✅ Done: {STAGE_LABELS.get(node_name, node_name)}")
            progress.progress((i + 1) / len(STAGES))

    status.empty()
    progress.empty()
    st.success("✅ Analysis complete!")

    col1, col2 = st.columns(2)
    with col1:
        st.subheader("📡 ISR")
        st.write(result.get("isr_output", ""))
        st.subheader("💻 Cyber")
        st.write(result.get("cyber_output", ""))
        st.subheader("🚛 Logistics")
        st.write(result.get("logistics_output", ""))
    with col2:
        st.subheader("⚠️ Threat Assessment")
        st.write(result.get("threat_output", ""))
        st.subheader("📋 Mission Planning + Debate")
        st.write(result.get("planner_output", ""))
        st.subheader("⚖️ Governance")
        st.write(result.get("governance_output", ""))

    st.subheader("🎖️ Commander Brief")
    st.info(result.get("commander_brief", ""))

    decision = st.radio("Commander Decision", ["Approve", "Reject"])
    st.write("Decision:", decision)


Overwriting app.py


In [1]:
!streamlit run app.py

^C
